In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfTransformer

# Initialize a matrix for the 200x200 grid (40,000 cells) and 85 POI categories
# Use mesh_x and mesh_y (1-200) to create a flattened index
poi_matrix = np.zeros((40000, 85))
directory = "cell_POIcat.csv/"

for filename in os.listdir(directory):
    try:
        parts = filename.split(',')
        if len(parts) == 4:
            x, y, cat_id, count = map(int, parts)
           
            grid_idx = (x - 1) * 200 + (y - 1)
            poi_matrix[grid_idx, cat_id - 1] = count
    except ValueError:
        continue

poi_df = pd.DataFrame(poi_matrix)

In [ ]:
lda = LatentDirichletAllocation(n_components=5, random_state=42,
                                 learning_method='batch', max_iter=20)
zone_distributions = lda.fit_transform(poi_df)  
zone_cols = [f'zone_{i}_prob' for i in range(5)]
dist_df = pd.DataFrame(zone_distributions, columns=zone_cols)
poi_df = pd.concat([poi_df, dist_df], axis=1)
poi_df['grid_id'] = range(40000)
poi_df['dominant_zone'] = dist_df.values.argmax(axis=1)

poi_df[['grid_id', 'dominant_zone'] + zone_cols].to_parquet("grid_zone_features.parquet", index=False)

feature_names = [f"Cat_{i}" for i in range(1, 86)]
for topic_idx, topic in enumerate(lda.components_):
    top_features = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"Zone {topic_idx} Top POIs: {top_features}")

Zone 0 Top POIs: ['Cat_59', 'Cat_48', 'Cat_76', 'Cat_69', 'Cat_79']
Zone 1 Top POIs: ['Cat_4', 'Cat_40', 'Cat_48', 'Cat_13', 'Cat_69']
Zone 2 Top POIs: ['Cat_69', 'Cat_62', 'Cat_51', 'Cat_66', 'Cat_63']
Zone 3 Top POIs: ['Cat_81', 'Cat_84', 'Cat_75', 'Cat_63', 'Cat_79']
Zone 4 Top POIs: ['Cat_74', 'Cat_60', 'Cat_73', 'Cat_47', 'Cat_79']


In [ ]:
import os
import gzip

mobility_records = []
mob_file_csv = "yjmob100k-dataset1.csv"
mob_file_gz = "yjmob100k-dataset1.csv.gz"

if os.path.exists(mob_file_csv):
    open_func = open
    mob_file = mob_file_csv
elif os.path.exists(mob_file_gz):
    open_func = gzip.open
    mob_file = mob_file_gz
else:
    raise FileNotFoundError("Neither yjmob100k-dataset1.csv nor yjmob100k-dataset1.csv.gz found.")

with open_func(mob_file, 'rt') as f:
    for line in f:
        parts = line.strip().split(',')
        if len(parts) != 5:
            continue
        try:
            uid, day, x, y, time_slot = map(int, parts)
            grid_id = (x - 1) * 200 + (y - 1) 
            mobility_records.append({
                'uid': uid,
                'day': day,
                'x': x,
                'y': y,
                'time_slot': time_slot,
                'grid_id': grid_id
            })
        except ValueError:
            continue

mob_df = pd.DataFrame(mobility_records)

mob_df = mob_df.sort_values(['uid', 'day', 'time_slot']).reset_index(drop=True)

zone_features = poi_df[['grid_id', 'dominant_zone'] + zone_cols]
mob_df = mob_df.merge(zone_features, on='grid_id', how='left')

print(mob_df.head(10))
print(f"Total mobility events: {len(mob_df):,}")
print(f"Unique users: {mob_df['uid'].nunique():,}")
print(f"Zone NaN rate: {mob_df['dominant_zone'].isna().mean():.2%}")

   uid  day   x   y  time_slot  grid_id  dominant_zone  zone_0_prob  \
0    0    0  25  75         82     4874            0.0          0.2   
1    0    0  29  76         83     5675            0.0          0.2   
2    0    0  27  76         84     5275            0.0          0.2   
3    0    0  43  76         85     8475            0.0          0.2   
4    0    0   1  79         86       78            0.0          0.2   
5    0    0   2  79         86      278            0.0          0.2   
6    0    0   8  77         86     1476            0.0          0.2   
7    0    0   9  77         86     1676            0.0          0.2   
8    0    0  24  76         86     4675            0.0          0.2   
9    0    0  30  77         86     5876            0.0          0.2   

   zone_1_prob  zone_2_prob  zone_3_prob  zone_4_prob  
0          0.2          0.2          0.2          0.2  
1          0.2          0.2          0.2          0.2  
2          0.2          0.2          0.2          